# Sentiment Analysis with 🤗 `transformers` `pipeline`

This notebook uses the Hugging Face `pipeline()` helper to run sentiment analysis.
All models chosen here are **small/lightweight** so they download quickly (tens to a couple hundred MB, not multi-GB):

1. Default binary sentiment (`distilbert-base-uncased-finetuned-sst-2-english`, ~268 MB)
2. Getting scores for **all** labels instead of just the top one
3. Batch inference on multiple sentences at once
4. Fine-grained 1-5 star sentiment (`nlptown/bert-base-multilingual-uncased-sentiment`) — multilingual variation
5. Emotion classification as a related variation of "sentiment" (`j-hartmann/emotion-english-distilroberta-base`, ~330 MB)

> Tip: the very first run of each pipeline downloads the model once and caches it locally (`~/.cache/huggingface`). Subsequent runs are instant.

In [1]:
# If needed, uncomment to install dependencies
%pip install -q transformers torch --upgrade

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Default sentiment analysis (binary: POSITIVE / NEGATIVE)

Not passing a `model` argument makes `pipeline` fall back to its default lightweight model for the `sentiment-analysis` task: `distilbert-base-uncased-finetuned-sst-2-english`.

In [2]:
from transformers import pipeline

sentiment_classifier = pipeline("sentiment-analysis")

result = sentiment_classifier("I absolutely loved this movie, it was fantastic!")
print(result)

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5455.66it/s]


[{'label': 'POSITIVE', 'score': 0.9998805522918701}]


In [3]:
# Try a negative example
print(sentiment_classifier("The service was terrible and I want a refund."))

[{'label': 'NEGATIVE', 'score': 0.999351441860199}]


## 2. Variation: return scores for every label

Use `top_k=None` (replaces the deprecated `return_all_scores=True`) to see the probability for both POSITIVE and NEGATIVE.

In [4]:
sentiment_all_scores = pipeline("sentiment-analysis", top_k=None)

print(sentiment_all_scores("It's okay, not great but not bad either."))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4657.80it/s]


[[{'label': 'POSITIVE', 'score': 0.9985427856445312}, {'label': 'NEGATIVE', 'score': 0.0014572517247870564}]]


## 3. Variation: batch inference on a list of sentences

The pipeline accepts a list of strings and returns a list of results — much faster than looping and calling it once per sentence.

In [5]:
texts = [
    "This is the best purchase I've made all year.",
    "Worst customer support experience ever.",
    "The product is fine, does what it says.",
    "I can't believe how good this is!",
]

for text, res in zip(texts, sentiment_classifier(texts)):
    print(f"{res['label']:>8} ({res['score']:.2%})  ->  {text}")

POSITIVE (99.98%)  ->  This is the best purchase I've made all year.
NEGATIVE (99.98%)  ->  Worst customer support experience ever.
POSITIVE (99.98%)  ->  The product is fine, does what it says.
POSITIVE (99.96%)  ->  I can't believe how good this is!


## 4. Variation: fine-grained sentiment (1 to 5 stars, multilingual)

`nlptown/bert-base-multilingual-uncased-sentiment` predicts a star rating from 1 to 5 and works across several languages (English, Dutch, German, French, Italian, Spanish).

In [6]:
star_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

for text in [
    "This restaurant exceeded all my expectations!",
    "Le service était moyen, rien de spécial.",
    "El producto llegó roto, muy decepcionado.",
]:
    print(star_sentiment(text), "->", text)

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2689.16it/s]


[{'label': '5 stars', 'score': 0.91909259557724}] -> This restaurant exceeded all my expectations!
[{'label': '2 stars', 'score': 0.5388386249542236}] -> Le service était moyen, rien de spécial.
[{'label': '1 star', 'score': 0.8468674421310425}] -> El producto llegó roto, muy decepcionado.


## 5. Variation: emotion classification

A close cousin of sentiment analysis: instead of just positive/negative, this model classifies text into emotions like joy, anger, sadness, fear, surprise, disgust, and neutral.

In [ ]:
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
)

for text in [
    "I am so excited for the trip tomorrow!",
    "I just lost my keys again, this is so frustrating.",
    "I'm scared about the exam results.",
]:
    scores = emotion_classifier(text)[0]
    top = max(scores, key=lambda x: x["score"])
    print(f"{top['label']:>10} ({top['score']:.2%})  ->  {text}")

## Summary

| Variation | Model | Approx. size | Output |
|---|---|---|---|
| Default binary | distilbert-base-uncased-finetuned-sst-2-english | ~268 MB | POSITIVE / NEGATIVE |
| All scores | same as above, `top_k=None` | — | scores for every label |
| Batch | same as above | — | list of results |
| Star rating | nlptown/bert-base-multilingual-uncased-sentiment | ~650 MB | 1-5 stars, multilingual |
| Emotion | j-hartmann/emotion-english-distilroberta-base | ~330 MB | joy/anger/sadness/... |

If you want to keep downloads minimal, stick to sections 1-3 which only need the ~268 MB default model.